<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/step3_text_feature_extract/extract_feature_openai_clip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ref

- https://github.com/openai/CLIP
- https://github.com/mlfoundations/open_clip


In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !cp -r "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014" "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/code/preprocessing/data/baby/2014"

In [ ]:
!cp -r "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/code/preprocessing/data" "./data"

In [ ]:
PATH = "./data/baby/2014"

In [ ]:
df = pd.read_parquet(
    os.path.join(PATH, "step3_text_feature_extract", "df_meta.preprocessed.parquet")
)

In [ ]:
df.head(3)

## Text Feature Extraction


In [ ]:
df.info()

In [ ]:
!pip install ftfy regex tqdm

In [ ]:
!python --version

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import clip
import torch

In [ ]:
clip.available_models()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-L/14", device=device)

print(f"CUDA khả dụng: {torch.cuda.is_available()}")
print(f"Tên GPU: {torch.cuda.get_device_name(device)}")
print("Model đã load thành công trên GPU local!")

In [ ]:
sentences = df["sentences"].tolist()

In [ ]:
sentences[:3]

In [ ]:
from tqdm.notebook import tqdm  # Thêm dòng này để import tqdm

print("Bắt đầu Encode... Vui lòng đợi trong giây lát.")

batch_size = 64  # Bạn có thể điều chỉnh kích thước batch này tùy theo dung lượng GPU của mình
text_embeddings_list = []

for i in tqdm(range(0, len(sentences), batch_size), desc="Processing batches"):
    batch_sentences = sentences[i : i + batch_size]
    text_tokens = clip.tokenize(batch_sentences, truncate=True).to(device)

    with torch.no_grad():
        batch_embeddings = model.encode_text(text_tokens)
    text_embeddings_list.append(batch_embeddings.cpu())

text_embeddings = torch.cat(text_embeddings_list);

assert text_embeddings.shape[0] == df.shape[0]
save_path = os.path.join(PATH, "step3_text_feature_extract", "text_feat.clip.npy")
# Lưu trực tiếp tensor đã chuyển về CPU
np.save(save_path, text_embeddings.numpy())

print(f"Done! File đã được lưu tại: {save_path}")

In [ ]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/AmazonData"

In [ ]:
!cp "./data/baby/2014/step3_text_feature_extract/text_feat.clip.npy" "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/code/data/baby/text_feat.npy"

In [ ]:
!cp "./data/baby/2014/step3_text_feature_extract/text_feat.clip.npy" "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014/text_feat.clip.npy"

In [ ]:
!ls "{PATH}/2014"

In [ ]:
!cp "{PATH}/2014/downloaded_images.zip" "downloaded_images.zip"

In [ ]:
!unzip downloaded_images.zip

### Tạo file `image_feature.b`


In [ ]:
import os
import numpy as np
import array
from tqdm.notebook import tqdm  # Import tqdm for progress bar

In [ ]:
def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [ ]:
def extract_and_write_features(
    image_directory,
    output_file_path,
    existing_asins_set,
    feature_extractor_fn,
    expected_size,
):
    image_files = [
        f
        for f in os.listdir(image_directory)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    total_images = len(image_files)
    processed_count = 0
    added_count = 0
    already_existing_count = 0
    invalid_asin_length_count = 0
    size_mismatch_count = 0  # Biến mới: Đếm lỗi sai kích thước vector

    print(
        f"Tìm thấy {total_images} hình ảnh. Đang trích xuất với size kỳ vọng: {expected_size}..."
    )

    with open(output_file_path, "ab") as f_out:
        for image_file in tqdm(image_files, desc="Trích xuất đặc trưng", unit="ảnh"):
            asin = os.path.splitext(image_file)[0].strip()
            img_path = os.path.join(image_directory, image_file)

            # 1. Check độ dài ASIN
            if len(asin) != 10:
                invalid_asin_length_count += 1
                processed_count += 1
                continue

            # 2. Check trùng
            if asin in existing_asins_set:
                already_existing_count += 1
                processed_count += 1
                continue

            # 3. Trích xuất
            features = feature_extractor_fn(img_path)

            if features is not None:
                try:
                    # Chuyển về numpy array 1D
                    features = np.array(features).flatten().astype(np.float32)

                    # KIỂM TRA ĐỘ DÀI TRƯỚC KHI GHI (Cực kỳ quan trọng)
                    if len(features) != expected_size:
                        size_mismatch_count += 1
                        processed_count += 1
                        continue

                    # Ghi ASIN (10 bytes)
                    f_out.write(asin[:10].encode("utf-8"))

                    # Ghi đúng số lượng float của model đó ra bytes
                    f_out.write(features.tobytes())

                    added_count += 1
                    existing_asins_set.add(asin)
                    f_out.flush()  # Ghi ngay xuống đĩa
                except Exception as e:
                    print(f"Lỗi khi ghi ASIN '{asin}': {e}")
            else:
                print(f"Không trích xuất được ASIN '{asin}'.")

            processed_count += 1

    print("\n--- BÁO CÁO CHI TIẾT ---")
    print(f"Tổng số hình ảnh đã xử lý: {processed_count}")
    print(f"Số đặc trưng đã thêm mới: {added_count}")
    print(f"Số ASIN đã có sẵn (bỏ qua): {already_existing_count}")
    print(f"Số ASIN sai độ dài (không phải 10): {invalid_asin_length_count}")
    print(f"Số ảnh sai kích thước vector ({expected_size}): {size_mismatch_count}")

In [ ]:
def main_feature_extraction(
    output_binary_file_name, image_directory, model_fn, feature_size
):
    # 1. Load các ASIN cũ (Phải truyền đúng size của model đã dùng cho file đó)
    existing_asins = set()
    if os.path.exists(output_binary_file_name):
        print(f"Đang kiểm tra dữ liệu cũ (Size: {feature_size})...")
        for asin, _ in read_image_features(output_binary_file_name, feature_size):
            existing_asins.add(asin)

    # 2. Chạy trích xuất mới
    extract_and_write_features(
        image_directory, output_binary_file_name, existing_asins, model_fn, feature_size
    )


# --- VÍ DỤ SỬ DỤNG ---

# Nếu dùng VGG16:
# main_feature_extraction('vgg_features.b', 'download_image', get_vgg16_features, 4096)

# Nếu dùng ResNet50:
# main_feature_extraction('resnet_features.b', 'download_image', get_resnet_features, 2048)

In [ ]:
# --- Script entry point ---
if __name__ == "__main__":
    output_binary_file = "image_feature.b"
    image_dir = "downloaded_images"
    main_feature_extraction(output_binary_file, image_dir, get_clip_image_features, 768)

In [ ]:
!cp "/content/image_feature.b" "{PATH}/2014/image_feature.open_ai_clip.b"

### Kiểm tra file vừa tạo bằng hàm `readImageFeatures` của bạn


In [ ]:
import array

# Đọc và kiểm tra file 'image_feature.b' vừa tạo
print(f"\nKiểm tra nội dung file '{output_binary_file}' bằng hàm của bạn:")
count = 0
for asin, features in read_image_features(output_binary_file, 768):
    print(f"ASIN: {asin}, Kiểu dữ liệu: {type(features)}, Kích thước: {len(features)}")
    if count < 2:  # Chỉ in chi tiết 3 mục đầu để không quá dài
        print(f"  Một vài giá trị đầu tiên: {features[:5]}")
    count += 1
    if count >= 3:  # Giới hạn số lượng mục in ra
        print("... và nhiều mục khác.")
        break

print(f"Tổng số mục đã đọc: {count}")